# 🧠 03: Model Training
**GUARDIAN-NLP** | Trains the GuardianBERT multi-label classifier.

Run this notebook on a GPU (Google Colab T4 recommended).

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import yaml

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Load Config

In [ ]:
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

# For quick notebook run, reduce epochs
config['training']['num_epochs'] = 3
config['training']['batch_size'] = 8  # reduce if OOM
print('Config loaded:')
print(yaml.dump(config, default_flow_style=False))

## 2. Verify Data

In [ ]:
import pandas as pd
import os

for split in ['train', 'val', 'test']:
    path = f'../data/processed/{split}.csv'
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'{split}: {len(df):,} rows | columns: {df.columns.tolist()}')
    else:
        print(f'{split}: NOT FOUND — run preprocessing pipeline first.')

## 3. Initialize GuardianBERT Model

In [ ]:
from src.models.bert_classifier import GuardianBERT

model = GuardianBERT(
    num_labels=config['model']['num_labels'],
    dropout=config['model']['dropout'],
    model_name=config['model']['name'],
)
print(f'Model architecture:\n{model}')
print(f'\nTotal trainable parameters: {model.count_parameters():,}')

## 4. Verify Dataset & DataLoader

In [ ]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader
from src.models.dataset import ToxicDataset

tokenizer = BertTokenizer.from_pretrained(config['model']['name'])
train_path = '../data/processed/train.csv'

if os.path.exists(train_path):
    dataset = ToxicDataset(train_path, tokenizer, max_length=config['model']['max_length'])
    loader = DataLoader(dataset, batch_size=4, shuffle=True)
    
    batch = next(iter(loader))
    print(f'Batch keys: {list(batch.keys())}')
    print(f'input_ids shape: {batch["input_ids"].shape}')
    print(f'attention_mask shape: {batch["attention_mask"].shape}')
    print(f'labels shape: {batch["labels"].shape}')
    print(f'Sample labels: {batch["labels"][0]}')
    print(f'Pos weights: {dataset.get_pos_weights()}')
else:
    print('Train data not found.')

## 5. Run Training

In [ ]:
# Update paths for notebook relative location
config['data']['train_path'] = '../data/processed/train.csv'
config['data']['val_path'] = '../data/processed/val.csv'
config['data']['test_path'] = '../data/processed/test.csv'
config['inference']['model_path'] = '../models/checkpoints/best_model.pt'
config['inference']['tokenizer_path'] = '../models/checkpoints/tokenizer/'
config['logging']['report_dir'] = '../outputs/reports/'

from src.models.trainer import GuardianTrainer
trainer = GuardianTrainer(config)

if os.path.exists('../data/processed/train.csv'):
    trainer.train()
else:
    print('⚠️ No training data found. Run the preprocessing pipeline first.')